<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex03-first-model/Ex03_vibrating_mass.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023. Ch. 2 and 3 for the first model.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---


# Ex_03 · Four ways to predict a vibrating mass

A mass on a spring is pulled aside and released. You have five seconds of
measurements from a cheap sensor, and you know the physics. Predict the next
seven seconds.

You will build four models, from a textbook formula to a physics-informed
neural network, and score them all against the same question: **not how well
they fit the measurements, but how close they are to the truth.**

*Deep Learning for Engineering · Lecture 3 · about 90 minutes*

## 0 · Setup

Run this once. It fetches `Ex_3_core.py`, which holds the data, the network,
and the training loop — everything that is plumbing rather than physics.

Nothing is saved to disk, so there is nothing to configure and nothing to lose.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_3_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex03-first-model/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
REPO   = "RemusTeodorescu/dl-for-eng"
FOLDER = "exercises/Ex03-first-model"

import os, getpass, urllib.request, urllib.error

if not os.path.exists("Ex_3_core.py"):
    # The course repository is private. Make a token at GitHub > Settings >
    # Developer settings > Personal access tokens > Fine-grained tokens, with
    # read-only Contents access to this repository and nothing else. It is
    # never written into the notebook.
    _tok = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token (hidden): ").strip()
    os.environ["GITHUB_TOKEN"] = _tok
    _req = urllib.request.Request(
        f"https://api.github.com/repos/{REPO}/contents/{FOLDER}/Ex_3_core.py",
        headers={"Authorization": f"Bearer {_tok}",
                 "Accept": "application/vnd.github.raw",
                 "User-Agent": "dl-for-eng"})
    try:
        with urllib.request.urlopen(_req) as r, open("Ex_3_core.py", "wb") as f:
            f.write(r.read())
        print("fetched Ex_3_core.py")
    except urllib.error.HTTPError as e:
        os.environ.pop("GITHUB_TOKEN", None)
        raise SystemExit(f"GitHub said {e.code}. Check the token has not expired "
                         f"and grants read access to {REPO}, then run this again.")
else:
    print("Ex_3_core.py already here")

import numpy as np
import matplotlib.pyplot as plt
import torch
import Ex_3_core as core

print("ready")

## 1 · What you have, and what you want

The system is a mass, a spring and a damper:

$$m\,\ddot{x} + c\,\dot{x} + k\,x = 0$$

You know three things.

**The constants**, from the drawing and the suppliers' catalogues. They are
approximately right — the damper in particular is made to a tolerance and its
catalogue figure was measured somewhere that is not your laboratory.

**The initial conditions.** The mass was pulled to 50 mm and released from
rest, so $x(0) = 0.05$ m and $\dot{x}(0) = 0$. These are not measurements;
they are how the experiment was set up, and they are known exactly.

**The measurements.** Five seconds from a cheap displacement sensor. It has
random noise, and it was never zeroed, so everything it reports is shifted by a
small constant amount.

Run the cell below to see all of it.

In [ ]:
t_data, x_data = core.load_measurements()

print(f"constants:  m = {core.SYSTEM['mass_kg']} kg   "
      f"k = {core.SYSTEM['stiffness_N_m']} N/m   "
      f"c = {core.SYSTEM['damping_Ns_m']} Ns/m  (datasheet)")
print(f"released from x(0) = {core.X0_M} m at rest")
print(f"measurements: {len(t_data)} samples over {core.T_DATA_S:.0f} s")
print(f"prediction wanted out to {core.T_END_S:.0f} s")

core.plot_models([], title="the measurements, and what really happened")
plt.show()

The grey line is the truth. **You would never have this** — it is here only so
that the models can be scored honestly. Everything you build will use the black
dots and the physics, never the grey line.

Look at the dots against the grey line near the top of each swing. They sit
slightly high, every time. That is the sensor bias, and it will matter.

## 2 · Model 1 — the analytic solution

For this equation a closed-form solution exists. With
$\omega_n = \sqrt{k/m}$ and $\zeta = c/(2\sqrt{km})$, a mass released from
rest at $x_0$ moves as

$$x(t) = e^{-\zeta\omega_n t}\left[x_0\cos\omega_d t
        + \frac{\zeta\omega_n x_0}{\omega_d}\sin\omega_d t\right],
  \qquad \omega_d = \omega_n\sqrt{1-\zeta^2}$$

No measurements are used at all. No parameters are fitted. It is pure physics
plus the catalogue.

**This is also why the rest of the exercise exists.** Closed-form solutions are
available for a small minority of engineering problems — simple ODEs like this
one. Nearly everything you will meet later is a PDE, in an awkward geometry,
and has no formula. Then you either mesh it and solve it numerically, or you do
what the rest of this notebook does.

### Model 1 — the analytic solution

The two dimensionless numbers come straight from the catalogue constants, and
`core.free_response(t, damping)` is the formula above with the datasheet damping.
Nothing to fill in here; run the cell and read what kind of system this is.

**How every model is scored.** Three numbers appear under each model, all of
them RMS errors in millimetres (`core.rmse`: the square root of the mean
squared difference between two displacement series):

- **RMSE inside** — model against the *true* displacement, at the 5 s of times
  where the sensor recorded. Can it fit?
- **RMSE outside** — model against the *true* displacement, after the data
  ends, where nothing was measured. Can it predict?
- **RMSE vs measurements** — model against the *sensor readings* themselves,
  which are noisy and never zeroed. The readings are 1.80 mm from the truth,
  so a model that agrees with them better than that has fitted their errors.

The truth is the noise-free trajectory the exercise generated the data from; a
real experiment never has it, which is the point of the last question.

In [ ]:
omega_n = core.natural_frequency()
zeta    = core.damping_ratio()

def model_analytic(t):
    return core.free_response(t, core.SYSTEM["damping_Ns_m"])

print(f"omega_n = {omega_n:.3f} rad/s   ->  period {2*np.pi/omega_n:.3f} s")
print(f"zeta    = {zeta:.4f}            ->  lightly damped, it will ring")
print()
in_A, out_A = core.score(model_analytic, "1 analytic")

Around 1.3 mm wrong, inside the data and outside it alike — the error does not
grow with time, it is just always there. That is the signature of a model whose
*form* is right and whose *constants* are slightly wrong. The catalogue damping
is 13% away from the real damping, and 13% of a small number is a small number.

Now plot it.

In [ ]:
core.plot_models([("1 analytic", model_analytic, "#0f9d58")],
                 title="model 1 — physics, no measurements")
plt.show()

## 3 · Model 2 — regression on the measurements

Suppose you did not trust the catalogue, or did not have it. You can measure
the damping from the record itself.

The peaks of a decaying oscillation fall on the envelope $A e^{-\zeta\omega_n t}$.
Take logarithms and that is a straight line:

$$\ln x_{\text{peak}} = \ln A - \zeta\omega_n t$$

So fit a straight line to the log of the peak heights, and its slope is the
decay rate. The spacing between peaks gives the damped period. This is the
**logarithmic decrement**, and structural engineers use it on real buildings.

It is also a genuine two-parameter regression — the first fitting you do in
this course.

### Model 2 — a curve fitted to the peaks

`core.find_peaks` gives the indices of the peaks. A straight line fitted to
`log(peak heights)` against `peak times` gives the decay rate (minus the slope)
and the amplitude; the mean spacing of the peaks gives the damped frequency.
Nothing to fill in; run it and compare the fitted decay rate with the true one.

In [ ]:
peaks = core.find_peaks(t_data, x_data)
t_peak, x_peak = t_data[peaks], x_data[peaks]
print(f"{len(peaks)} peaks at t = {np.round(t_peak, 2)}")
print(f"heights      = {np.round(x_peak*1000, 2)} mm")

slope, intercept = np.polyfit(t_peak, np.log(x_peak), 1)
decay_rate = -slope
amplitude  = np.exp(intercept)
omega_d    = 2*np.pi / np.mean(np.diff(t_peak))

def model_regression(t):
    return amplitude * np.exp(-decay_rate * np.asarray(t)) * np.cos(omega_d * np.asarray(t))

true_decay = core._C_TRUE / (2 * core.SYSTEM["mass_kg"])
print(f"\nfitted decay rate {decay_rate:.4f} 1/s   (true {true_decay:.4f})")
print(f"fitted omega_d    {omega_d:.3f} rad/s")
print()
in_B, out_B = core.score(model_regression, "2 regression")

Worse than the formula, and the fitted decay rate comes out low. Why?

**The sensor bias.** Every peak is reported about 1.5 mm higher than it really
was. The later peaks are small, so 1.5 mm is a large fraction of them; the early
peaks are big, so it is a small fraction. The bias therefore flattens the
apparent decay, and the fit reports a system that loses energy more slowly than
it does.

The regression has no way to know this. It sees numbers and fits them.

In [ ]:
core.plot_models([("1 analytic", model_analytic, "#0f9d58"),
                  ("2 regression", model_regression, "#f4a300")],
                 title="model 2 — fitted from the measurements")
plt.show()

## 4 · Model 3 — a neural network

Now forget the physics entirely and do what a machine-learning course would do:
train a network to map time to displacement, using nothing but the measurements.

The network is built for you in `core.build_network()`. It has about 9,500
parameters — against the analytic model's zero and the regression's two.

`core.train(...)` runs the training loop. Left as it is, with only `w_data` set,
it is an ordinary curve fit: minimise the difference between the network's
output and the measured points, and nothing else.

### TODO 1 — the network alone

Train on the measurements only. Set `w_data` to `1.0` and leave `w_physics`
and `w_initial` at `0.0`, then run. This takes about half a minute.

In [ ]:
net_plain = core.build_network()
print(f"{core.count_parameters(net_plain)} parameters\n")

# TODO 1 --- set the three weights ----------------------------------------
w_data    = 0.0
w_physics = 0.0
w_initial = 0.0
# ---------------------------------------------------------------------------
if w_data == w_physics == w_initial == 0.0:
    raise SystemExit("Set the weights in TODO 1 above, then run this cell again.")
core.train(net_plain, t_data, x_data,
           w_data=w_data, w_physics=w_physics, w_initial=w_initial,
           epochs=6000)

def model_network(t):
    return core.predict(net_plain, t)

print()
in_C, out_C = core.score(model_network, "3 neural network")
print(f"\nRMSE vs measurements:            {core.rmse(model_network(t_data), x_data)*1000:.3f} mm")
print(f"the measurements are wrong by     {core.measurement_error()*1000:.3f} mm RMS (noise and bias, against the truth)")

Read those last two numbers carefully.

The network agrees with the measurements **more closely than the measurements
are correct.** It has fitted the noise and the bias along with the signal,
because it has no way to tell them apart. Nine thousand parameters and no
opinion about physics will do that.

Now plot it, and look at what happens the moment the data stops.

In [ ]:
core.plot_models([("3 neural network", model_network, "#d94f2b")],
                 title="model 3 — measurements only")
plt.show()

This is the important picture in the exercise.

Inside the shaded region the network is excellent — better than anything else
you have built. Outside it, the prediction is worthless. It does not know the
mass keeps oscillating, because nothing ever told it, and nothing in its
training gave it a reason to think time continues to behave the same way.

**A model can fit the data best and still be the worst model.** There is no way
to discover this from the training error alone.

## 5 · Model 4 — a physics-informed neural network

Same network. Same measurements. One thing added.

As well as asking the network to pass near the measured points, we ask it to
satisfy the differential equation. At several hundred times spread across the
**whole** interval — including the seven seconds where no measurement exists —
we compute

$$r(t) = m\,\ddot{x} + c\,\dot{x} + k\,x$$

from the network's own output, and add $r^2$ to the loss. Those times are
called **collocation points**, and PyTorch gives us $\dot{x}$ and $\ddot{x}$
exactly by differentiating the network, so no mesh and no finite differences are
involved.

We also add the initial conditions, $x(0) = 0.05$ and $\dot{x}(0) = 0$. They
matter more than they look: the equation $m\ddot{x} + c\dot{x} + kx = 0$ is
satisfied perfectly by $x \equiv 0$, so physics alone does not pick out your
oscillation. The initial conditions are what make the problem well-posed.

The loss is now three terms with three weights:

$$L = w_{\text{data}}L_{\text{data}} + w_{\text{phys}}L_{\text{phys}}
    + w_{\text{init}}L_{\text{init}}$$

### TODO 2 — switch the physics on

Same call, same network size. Set `w_data` to `1.0` again, and `w_physics` and
`w_initial` to `10.0`, then run.

In [ ]:
net_pinn = core.build_network()

# TODO 2 --- set the three weights ----------------------------------------
w_data    = 0.0
w_physics = 0.0
w_initial = 0.0
# ---------------------------------------------------------------------------
if w_data == w_physics == w_initial == 0.0:
    raise SystemExit("Set the weights in TODO 2 above, then run this cell again.")
core.train(net_pinn, t_data, x_data,
           w_data=w_data, w_physics=w_physics, w_initial=w_initial,
           epochs=6000)

def model_pinn(t):
    return core.predict(net_pinn, t)

print()
in_D, out_D = core.score(model_pinn, "4 PINN")
print(f"\nRMSE vs measurements:            {core.rmse(model_pinn(t_data), x_data)*1000:.3f} mm")
print(f"the measurements are wrong by     {core.measurement_error()*1000:.3f} mm RMS (noise and bias, against the truth)")

In [ ]:
core.plot_models([("3 neural network", model_network, "#d94f2b"),
                  ("4 PINN", model_pinn, "#0f9d58")],
                 title="model 4 — measurements and physics together")
plt.show()

Compare those two numbers with the same two from model 3.

Model 3 agreed with the measurements more closely than they deserve. Model 4
disagrees with them by roughly the amount they are actually wrong — it has
declined to follow the noise and the bias, because doing so would break the
differential equation. It fits the measurements *worse* and is *closer to the
truth*.

This is why physics in the loss is not only about extrapolation. It is a
statement about which parts of your data to believe.

## 6 · The experiment — how much should you believe the physics?

`w_physics` is a choice, and it is the main thing you tune in a PINN. Too low
and you have model 3 again. Too high and the measurements stop mattering at all.

Run the sweep below. It trains four models and takes two or three minutes.

In [ ]:
weights = [0.0, 1.0, 10.0, 100.0]
sweep = []

for w in weights:
    net = core.build_network()
    core.train(net, t_data, x_data, w_data=1.0, w_physics=w,
               w_initial=10.0 if w > 0 else 0.0, epochs=6000, report=False)
    f = lambda t, n=net: core.predict(n, t)
    inside, outside = core.score(f)
    sweep.append((w, inside, outside, core.rmse(f(t_data), x_data)))
    print(f"  w_physics = {w:<6}  RMSE inside {inside*1000:6.3f} mm   "
          f"RMSE outside {outside*1000:7.3f} mm   "
          f"RMSE vs measurements {sweep[-1][3]*1000:6.3f} mm")

print(f"\nfor reference, the measurements are {core.measurement_error()*1000:.3f} mm RMS from the truth "
      f"(sensor noise and an un-zeroed bias): a model closer to them than that has fitted their errors")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
w_plot = [max(w, 0.1) for w, *_ in sweep]     # 0 shown at the left edge
ax.plot(w_plot, [s[3]*1000 for s in sweep], "o-", color="#d94f2b",
        label="disagreement with the measurements")
ax.plot(w_plot, [s[1]*1000 for s in sweep], "o-", color="#0f9d58",
        label="error against the truth")
ax.axhline(core.measurement_error()*1000, color="#9aa5b1", ls="--",
           label="how wrong the measurements are")
ax.set_xscale("log"); ax.set_xlabel("physics weight"); ax.set_ylabel("RMSE [mm]")
ax.set_xticks(w_plot); ax.set_xticklabels([str(w) for w, *_ in sweep])
ax.legend(frameon=False, fontsize=9); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

The two curves go opposite ways. As you trust the physics more, the model agrees
with the measurements *less* and with the truth *more* — and the disagreement
settles at about the level the measurements are wrong by. It has found the
signal and left the errors behind.

### TODO 3 — the extreme case

Set `w_data` to `0.0`, and `w_physics` and `w_initial` to `100.0`. The network
then never sees a single measurement: it has only the differential equation and
the two initial conditions.

Predict what will happen before you run it.

In [ ]:
net_nodata = core.build_network()

# TODO 3 --- set the three weights ----------------------------------------
w_data    = 0.0
w_physics = 0.0
w_initial = 0.0
# ---------------------------------------------------------------------------
if w_data == w_physics == w_initial == 0.0:
    raise SystemExit("Set the weights in TODO 3 above, then run this cell again.")
core.train(net_nodata, t_data, x_data,
           w_data=w_data, w_physics=w_physics, w_initial=w_initial,
           n_collocation=800,  # with no data to anchor it, check the physics in more places
           epochs=6000)

def model_nodata(t):
    return core.predict(net_nodata, t)

print()
in_E, out_E = core.score(model_nodata, "5 PINN, no measurements")

core.plot_models([("5 physics + initial conditions only", model_nodata, "#7b5ea7")],
                 title="model 5 — no measurements were used")
plt.show()

It works. The network has solved the differential equation.

Look at where it landed: about the same error as **model 1**, the textbook
formula. That is not a coincidence — both of them used the catalogue damping,
which is 13% wrong, and neither had anything to correct it with. They are two
ways of computing the same slightly-wrong physics, one with a pen and one with
9,473 parameters.

Model 4 beat them both, and now you can say exactly why. The measurements never
taught it the shape of the motion — the physics did that. What the measurements
contributed was a correction to the constant the catalogue got wrong.

**Try one more thing:** set `w_initial = 0` as well, so only the physics
remains. The network will collapse to $x \\equiv 0$, because that satisfies
$m\\ddot{x} + c\\dot{x} + kx = 0$ perfectly. A differential equation without
its initial conditions does not have one answer, and no amount of training will
invent one.


## 7 · Results

Everything in one table.

In [ ]:
rows = [
    ["1 analytic (catalogue constants)", 0, f"{in_A*1000:.3f}", f"{out_A*1000:.3f}"],
    ["2 regression on peaks",            2, f"{in_B*1000:.3f}", f"{out_B*1000:.3f}"],
    ["3 neural network",  core.count_parameters(net_plain), f"{in_C*1000:.3f}", f"{out_C*1000:.3f}"],
    ["4 PINN",            core.count_parameters(net_pinn),  f"{in_D*1000:.3f}", f"{out_D*1000:.3f}"],
    ["5 PINN, no measurements", core.count_parameters(net_nodata), f"{in_E*1000:.3f}", f"{out_E*1000:.3f}"],
]
table = core.error_table(rows, ["model", "fitted parameters",
                                "RMSE inside [mm]", "RMSE outside [mm]"])
print(table)

core.plot_models([("1 analytic",   model_analytic,   "#0f9d58"),
                  ("2 regression", model_regression, "#f4a300"),
                  ("3 network",    model_network,    "#d94f2b"),
                  ("4 PINN",       model_pinn,       "#1f6feb")],
                 title="all four models")
plt.show()

## 8 · Your report

Answer these in your own words. Two or three sentences each is plenty — this is
about whether you can explain the result, not about length.

1. Model 3 agreed with the measurements better than any other model, and was
   the worst prediction. Explain to an engineer who has not done this exercise
   how that is possible, and what they should look at instead.

2. The sensor had a constant bias. Which of the five models were damaged by it,
   which were not, and why?

3. Model 5 used no measurements at all and still worked. What did the
   measurements contribute in model 4, then?

4. From your sweep, which physics weight would you choose, and what would you
   need to know about a *real* problem — where there is no truth to check
   against — to make that choice?

5. The analytic model was cheap and good. Under what circumstances would you
   reach for a PINN instead, and when would you not bother?

Write your answers in the cell below, then run it to produce the report.

In [ ]:
answers = {
    "1 fitting versus predicting": """

    """,
    "2 the effect of the sensor bias": """

    """,
    "3 what the measurements contributed": """

    """,
    "4 choosing the physics weight": """

    """,
    "5 when a PINN is worth it": """

    """,
}

from datetime import date
report = [f"# Ex_03 report", f"", f"Date: {date.today()}", "", "## Results", "",
          "```", table, "```", ""]
for q, a in answers.items():
    report += [f"## {q}", "", a.strip() or "_(not answered)_", ""]
report = "\n".join(report)

print(report)
print("-" * 70)
missing = sum(1 for a in answers.values() if not a.strip())
print(f"{len(report.split())} words, {missing} question(s) still unanswered")

with open("Ex03_report.md", "w", encoding="utf-8") as fh:
    fh.write(report)
try:
    from google.colab import files
    files.download("Ex03_report.md")
except ImportError:
    print("written to Ex03_report.md beside this notebook")

---

### What this exercise was for

You have met, in miniature, the argument the rest of the course builds on.

A neural network with no physics is an interpolator. It is superb between your
data points and says nothing trustworthy beyond them, and it cannot tell signal
from sensor error. Adding the governing equation to the loss changes what the
network *is*: it becomes something that solves your physics and uses data to
pin down what the physics alone leaves open.

Here the equation was an ODE with a known solution, so you could check every
answer. From Lecture 7 onward the equations are PDEs in real geometries, where
no formula exists and a mesh is expensive — and that is where this stops being
a demonstration and starts being useful.